In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

data.en1  data.en4  data.ta1  data.ta4	tamil_english_sentences.csv
data.en2  data.en5  data.ta2  data.ta5
data.en3  data.en6  data.ta3  data.ta6


# **SMT TRAINING**

In [ ]:
# Clone the official Moses repository (contains all scripts)
!git clone https://github.com/moses-smt/mosesdecoder.git

# Verify the folder exists now
!ls mosesdecoder/scripts/tokenizer/

Cloning into 'mosesdecoder'...
remote: Enumerating objects: 148472, done.
remote: Counting objects: 100% (900/900), done.
remote: Compressing objects: 100% (375/375), done.
remote: Total 148472 (delta 571), reused 806 (delta 520), pack-reused 147572 (from 1)
Receiving objects: 100% (148472/148472), 129.99 MiB | 19.21 MiB/s, done.
Resolving deltas: 100% (114597/114597), done.
basic-protected-patterns	 normalize-punctuation.perl
deescape-special-chars.perl	 pre-tok-clean.perl
deescape-special-chars-PTB.perl  pre_tokenize_cleaning.py
delete-long-words.perl		 pre-tokenizer.perl
detokenizer.perl		 remove-non-printing-char.perl
escape-special-chars.perl	 replace-unicode-punctuation.perl
lowercase.perl			 tokenizer.perl
mosestokenizer			 tokenizer_PTB.perl


In [ ]:
import os

# Path to your dataset folder
path = "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

# Verify Tamil/English files
!ls "$path"

data.en1  data.en4  data.ta1  data.ta4	tamil_english_sentences.csv
data.en2  data.en5  data.ta2  data.ta5
data.en3  data.en6  data.ta3  data.ta6


In [ ]:
# This entire block runs in bash (not Python)
path="/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

# Tokenize Tamil
!perl mosesdecoder/scripts/tokenizer/tokenizer.perl -l ta < "$path/all_data.ta" > "$path/train.tok.ta"

# Tokenize English
!perl mosesdecoder/scripts/tokenizer/tokenizer.perl -l en < "$path/all_data.en" > "$path/train.tok.en"

# Confirm
!echo "✅ Tokenization complete!"
!wc -l "$path/train.tok.ta" "$path/train.tok.en"


Tokenizer Version 1.1
Language: ta
Number of threads: 1
Tokenizer Version 1.1
Language: en
Number of threads: 1
✅ Tokenization complete!
   289451 /content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/train.tok.ta
   289451 /content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/train.tok.en
   578902 total


In [ ]:
!perl /content/mosesdecoder/scripts/training/clean-corpus-n.perl \
"$path/train.tok" ta en "$path/train.clean" 1 80

clean-corpus.perl: processing /content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/train.tok.ta & .en to /content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/train.clean, cutoff 1-80, ratio 9
..........(100000)..........(200000)........
Input sentences: 289451  Output sentences:  241793


In [ ]:
# Path to your FastAlign build
EXT_BIN="/content/fast_align/build"

# Verify fast_align binaries exist
!ls "$EXT_BIN"

atools		CMakeFiles	     fast_align      Makefile
CMakeCache.txt	cmake_install.cmake  force_align.py


In [ ]:
!mkdir -p /content/external_bin
!ln -s "$EXT_BIN/fast_align" /content/external_bin/fast_align
!ln -s "$EXT_BIN/atools" /content/external_bin/atools

# Verify
!ls /content/external_bin

atools	fast_align


In [ ]:
# Path to working folder for model output
path="/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"
EXT_BIN="/content/fast_align/build"   # we’ll install this next

# Clone and build FastAlign (used for word alignment)
!git clone https://github.com/clab/fast_align.git
%cd fast_align
!mkdir build && cd build && cmake .. && make -j
%cd /content

Cloning into 'fast_align'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 213 (delta 32), reused 26 (delta 26), pack-reused 172 (from 1)
Receiving objects: 100% (213/213), 62.05 KiB | 858.00 KiB/s, done.
Resolving deltas: 100% (115/115), done.
/content/fast_align
CMake Warning (dev) at CMakeLists.txt:1 (project):
  cmake_minimum_required() should be called prior to this top-level project()
  call.  Please see the cmake-commands(7) manual for usage documentation of
  both commands.
This warning is for project developers.  Use -Wno-dev to suppress it.

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detect

# **GIZA++**

In [ ]:
# Clone GIZA++ and supporting tools
!git clone https://github.com/moses-smt/giza-pp.git
!git clone https://github.com/moses-smt/mgiza.git

Cloning into 'giza-pp'...
remote: Enumerating objects: 328, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 328 (delta 5), reused 9 (delta 3), pack-reused 301 (from 1)
Receiving objects: 100% (328/328), 314.48 KiB | 2.02 MiB/s, done.
Resolving deltas: 100% (212/212), done.
Cloning into 'mgiza'...
remote: Enumerating objects: 1053, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 1053 (delta 0), reused 3 (delta 0), pack-reused 1042 (from 1)
Receiving objects: 100% (1053/1053), 1.30 MiB | 5.22 MiB/s, done.
Resolving deltas: 100% (644/644), done.


In [ ]:
# Compile giza-pp
%cd giza-pp
!make -j
%cd ..

# Compile mkcls (word clustering)
%cd giza-pp/mkcls
!make -j
%cd /content

/content/giza-pp
make -C GIZA++-v2
make -C mkcls-v2
make[1]: Entering directory '/content/giza-pp/mkcls-v2'
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c GDAOptimization.cpp -o GDAOptimization.o
make[1]: Entering directory '/content/giza-pp/GIZA++-v2'
mkdir optimized/
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c HCOptimization.cpp -o HCOptimization.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c Problem.cpp -o Problem.o
g++   -Wall -Wno-parentheses -O3 -funroll-loops -DNDEBUG -DWORDINDEX_WITH_4_BYTE -DBINARY_SEARCH_FOR_TTABLE  -c Parameter.cpp -o optimized/Parameter.o
g++   -Wall -Wno-parentheses -O3 -funroll-loops -DNDEBUG -DWORDINDEX_WITH_4_BYTE -DBINARY_SEARCH_FOR_TTABLE  -c myassert.cpp -o optimized/myassert.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops -c IterOptimization.cpp -o IterOptimization.o
g++   -Wall -Wno-parentheses -O3 -funroll-loops -DNDEBUG -DWORDINDEX_WITH_4_BYTE -DBINARY_SEARCH_FOR_TTABLE  -c Perplexity.cpp -o optimized/Perplexity.o
g++ -Wall -W -DNDEBUG -O3 -funroll-loops 

In [ ]:
# Verify where the compiled binaries are
!ls -l /content/giza-pp/GIZA++-v2 | head
!ls -l /content/giza-pp/mkcls-v2 | head

# Create an external_bin folder and copy the needed tools
!mkdir -p /content/external_bin
!cp /content/giza-pp/GIZA++-v2/GIZA++ /content/external_bin/
!cp /content/giza-pp/GIZA++-v2/snt2cooc.out /content/external_bin/
!cp /content/giza-pp/mkcls-v2/mkcls /content/external_bin/

# Confirm they’re there
!ls -l /content/external_bin

total 2404
-rw-r--r-- 1 root root    1326 Nov  8 05:00 alignment.cpp
-rw-r--r-- 1 root root    5518 Nov  8 05:00 alignment.h
-rw-r--r-- 1 root root    1500 Nov  8 05:00 AlignTables.cpp
-rw-r--r-- 1 root root    4100 Nov  8 05:00 AlignTables.h
-rw-r--r-- 1 root root    2996 Nov  8 05:00 Array2.h
-rw-r--r-- 1 root root    1818 Nov  8 05:00 Array4.h
-rw-r--r-- 1 root root     106 Nov  8 05:00 Array.h
-rw-r--r-- 1 root root    3838 Nov  8 05:00 ATables.cpp
-rw-r--r-- 1 root root    6067 Nov  8 05:00 ATables.h
total 1680
-rw-r--r-- 1 root root   8410 Nov  8 05:00 Array.h
-rw-r--r-- 1 root root   6202 Nov  8 05:00 FixedArray.h
-rw-r--r-- 1 root root   1237 Nov  8 05:00 FlexArray.h
-rw-r--r-- 1 root root   3560 Nov  8 05:00 GDAOptimization.cpp
-rw-r--r-- 1 root root   1711 Nov  8 05:00 GDAOptimization.h
-rw-r--r-- 1 root root  20408 Nov  8 05:02 GDAOptimization.o
-rw-r--r-- 1 root root   2110 Nov  8 05:00 general.cpp
-rw-r--r-- 1 root root   1622 Nov  8 05:00 general.h
-rw-r--r-- 1 root root 

In [ ]:
path="/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

!mkdir -p "$path/working"

!perl /content/mosesdecoder/scripts/training/train-model.perl \
  --root-dir "$path/working" \
  --corpus "$path/train.clean" \
  --f ta --e en \
  --alignment grow-diag-final-and \
  --reordering msd-bidirectional-fe \
  --external-bin-dir /content/external_bin \
  --lm 0:3:/dev/null:8


Using SCRIPTS_ROOTDIR: /content/mosesdecoder/scripts
Using single-thread GIZA
using pigz 
(1) preparing corpus @ Sat Nov  8 05:06:13 UTC 2025
Executing: mkdir -p /content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/working/corpus
(1.0) selecting factors @ Sat Nov  8 05:06:13 UTC 2025
(1.1) running mkcls  @ Sat Nov  8 05:06:13 UTC 2025
/content/external_bin/mkcls -c50 -n2 -p/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/train.clean.ta -V/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/working/corpus/ta.vcb.classes opt
Executing: /content/external_bin/mkcls -c50 -n2 -p/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/train.clean.ta -V/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/working/corpus/ta.vcb.classes opt

***** 2 runs. (algorithm:TA)*****
;KategProblem:cats: 50   words: 122114

start-costs: MEAN: 1.63273e+08 (1.63248e+08-1.63299e+08

In [ ]:
path="/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

# Make smaller subsets (100k sentences)
!head -n 100000 "$path/train.clean.ta" > "$path/train.small.ta"
!head -n 100000 "$path/train.clean.en" > "$path/train.small.en"

# Train Moses on 100k subset
!perl /content/mosesdecoder/scripts/training/train-model.perl \
  --root-dir "$path/working_small" \
  --corpus "$path/train.small" \
  --f ta --e en \
  --alignment grow-diag-final-and \
  --reordering msd-bidirectional-fe \
  --external-bin-dir /content/external_bin \
  --lm 0:3:/dev/null:8


Using SCRIPTS_ROOTDIR: /content/mosesdecoder/scripts
Using single-thread GIZA
using pigz 
(1) preparing corpus @ Sat Nov  8 05:17:29 UTC 2025
Executing: mkdir -p /content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/working_small/corpus
(1.0) selecting factors @ Sat Nov  8 05:17:29 UTC 2025
(1.1) running mkcls  @ Sat Nov  8 05:17:29 UTC 2025
/content/external_bin/mkcls -c50 -n2 -p/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/train.small.ta -V/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/working_small/corpus/ta.vcb.classes opt
Executing: /content/external_bin/mkcls -c50 -n2 -p/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/train.small.ta -V/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/working_small/corpus/ta.vcb.classes opt

***** 2 runs. (algorithm:TA)*****
;KategProblem:cats: 50   words: 86637

start-costs: MEAN: 6.12707e+07 (6.1267

In [ ]:
path="/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

# Prepare combined parallel text for FastAlign
!paste "$path/train.small.ta" "$path/train.small.en" | tr '\t' '|' > "$path/parallel.txt"

# Forward alignment (ta→en)
!cat "$path/parallel.txt" | /content/external_bin/fast_align -i - -d -o -v > "$path/forward.align"

# Reverse alignment (en→ta)
!cat "$path/parallel.txt" | /content/external_bin/fast_align -i - -d -o -v -r > "$path/reverse.align"

# Symmetrize alignments
!/content/external_bin/atools -i "$path/forward.align" -j "$path/reverse.align" -c grow-diag-final-and > "$path/symmetrized.align"

!head "$path/symmetrized.align"


ARG=i
ARG=d
ARG=o
ARG=v
Can't read -
INITIAL PASS 
expected target length = source length * -nan
ITERATION 1
Can't read -
ARG=i
ARG=d
ARG=o
ARG=v
ARG=r
Can't read -
INITIAL PASS 
expected target length = source length * -nan
ITERATION 1
Can't read -


In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

# Read both files safely in UTF-8
with open(f"{path}/train.small.ta", encoding="utf-8") as f1, \
     open(f"{path}/train.small.en", encoding="utf-8") as f2:
    tamil_lines = [x.strip() for x in f1.readlines()]
    eng_lines = [x.strip() for x in f2.readlines()]

# Ensure same number of lines
pairs = zip(tamil_lines, eng_lines)
with open(f"{path}/parallel.txt", "w", encoding="utf-8") as f:
    for ta, en in pairs:
        f.write(f"{ta} ||| {en}\n")

print("✅ parallel.txt created safely.")
!head -n 3 "$path/parallel.txt"


✅ parallel.txt created safely.
ராஜாவாகிய ஆகாஸ ் அரசாளும ் போது தம ் முடைய பாதகத ் தினால ் எறிந ் துபோட ் ட சகல பணிமுட ் டுகளையும ் முஸ ் திப ் பாக ் கிப ் பரிசுத ் தம ் பண ் ணினோம ் ; இதோ , அவைகள ் கர ் த ் தரின ் ஆலயத ் திற ் கு முன ் பாக இருக ் கிறது என ் றார ் கள ் . ||| moreover all the vessels , which king ahaz in his reign did cast away in his transgression , have we prepared and sanctified , and , behold , they are before the altar of the lord .
சர ் வதேச நாணய நிதியம ் இலங ் கைக ் கு கடன ் வழங ் கினால ் இதே போன ் ற நிபந ் தனைகள ் திணிக ் கப ் படும ் . ||| similar conditions will be imposed if the sri lankan government is given an imf loan .
தற ் போது அதற ் கு எதிராக வாதாடுகிறார ் சர ் வதேச சட ் டத ் தை செயல ் படுத ் துவதற ் குப ் பதிலாக புதிய சட ் டம ் உருவாக ் கப ் பட ் டு நிறுவப ் பட வேண ் டும ் என ் று எழுதுகிறார ் . ||| now kornelius argues the opposite instead of enforcing the adherence to international law , new laws would now have to be devised and established .


In [ ]:
!file -i "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/train.small.ta"

/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/train.small.ta: text/plain; charset=utf-8


In [ ]:
import unicodedata

path = "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

fixed_lines = []
with open(f"{path}/train.small.ta", encoding="utf-8") as f:
    for line in f:
        fixed_lines.append(unicodedata.normalize("NFC", line.strip()))

# Save fixed Tamil
with open(f"{path}/train.small.fixed.ta", "w", encoding="utf-8") as f:
    for line in fixed_lines:
        f.write(line + "\n")

print("✅ Tamil text normalized and saved as train.small.fixed.ta")
!head -n 3 "$path/train.small.fixed.ta"

✅ Tamil text normalized and saved as train.small.fixed.ta
ராஜாவாகிய ஆகாஸ ் அரசாளும ் போது தம ் முடைய பாதகத ் தினால ் எறிந ் துபோட ் ட சகல பணிமுட ் டுகளையும ் முஸ ் திப ் பாக ் கிப ் பரிசுத ் தம ் பண ் ணினோம ் ; இதோ , அவைகள ் கர ் த ் தரின ் ஆலயத ் திற ் கு முன ் பாக இருக ் கிறது என ் றார ் கள ் .
சர ் வதேச நாணய நிதியம ் இலங ் கைக ் கு கடன ் வழங ் கினால ் இதே போன ் ற நிபந ் தனைகள ் திணிக ் கப ் படும ் .
தற ் போது அதற ் கு எதிராக வாதாடுகிறார ் சர ் வதேச சட ் டத ் தை செயல ் படுத ் துவதற ் குப ் பதிலாக புதிய சட ் டம ் உருவாக ் கப ் பட ் டு நிறுவப ் பட வேண ் டும ் என ் று எழுதுகிறார ் .


# **VER 2.0**

# **MERGING THE DATASETS**

In [ ]:
import glob

# Output paths inside the same folder
merged_ta = os.path.join(path, "all_data.ta")
merged_en = os.path.join(path, "all_data.en")

# Merge Tamil files
with open(merged_ta, "w", encoding="utf-8") as outfile:
    for fname in sorted(glob.glob(os.path.join(path, "data.ta*"))):
        with open(fname, encoding="utf-8") as infile:
            outfile.write(infile.read().strip() + "\n")

# Merge English files
with open(merged_en, "w", encoding="utf-8") as outfile:
    for fname in sorted(glob.glob(os.path.join(path, "data.en*"))):
        with open(fname, encoding="utf-8") as infile:
            outfile.write(infile.read().strip() + "\n")

print(" Merged successfully!")
print("Saved files:")
print(merged_ta)
print(merged_en)

# Optional: line count check
!wc -l "$merged_ta" "$merged_en"


✅ Merged successfully!
Saved files:
/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.ta
/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.en
   289451 /content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.ta
   289451 /content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.en
   578902 total


In [ ]:
!file -i "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.ta"
!file -i "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.en"


/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.ta: text/plain; charset=utf-8
/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.en: text/plain; charset=utf-8


In [ ]:
!wc -l "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.ta"
!wc -l "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.en"

289451 /content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.ta
289451 /content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.en


In [ ]:
!head -n 5 "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.ta"
!head -n 5 "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.en"


ராஜாவாகிய ஆகாஸ் அரசாளும்போது தம்முடைய பாதகத்தினால் எறிந்துபோட்ட சகல பணிமுட்டுகளையும் முஸ்திப்பாக்கிப் பரிசுத்தம்பண்ணினோம்; இதோ , அவைகள் கர்த்தரின் ஆலயத்திற்கு முன்பாக இருக்கிறது என்றார்கள் .
சர்வதேச நாணய நிதியம் இலங்கைக்கு கடன் வழங்கினால் இதே போன்ற நிபந்தனைகள் திணிக்கப்படும் .
தற்போது அதற்கு எதிராக வாதாடுகிறார் சர்வதேச சட்டத்தை செயல்படுத்துவதற்குப் பதிலாக புதிய சட்டம் உருவாக்கப்பட்டு நிறுவப்பட வேண்டும் என்று எழுதுகிறார் .
அமெரிக்காவின் மூன்றாம் பெரிய கார் தயாரிப்பு நிறுவனமான கிறைஸ்லர் வியாழனன்று நியூ யோர்க்கில் திவாலடைந்ததற்காக மனு செய்தது; அத்தியாயம் 11 ன் படி மறு சீரமைத்து வெளிவரும் வரை அது தன்னுடைய உற்பத்தி நிலையங்களை மூடும் என்றும் அறிவித்துள்ளது .
மேலும் இனைவிட்டு தலிபானால் வெளியேற்றப்பட்ட 1995 இல் இருந்து ஈரானில் கூடுதலாக வாழ்ந்துவந்துள்ளார் .
moreover all the vessels , which king ahaz in his reign did cast away in his transgression , have we prepared and sanctified , and , behold , they are before the altar of the lord .
similar conditions will be imposed if the sri lankan go

In [ ]:
!sed -n '100p' "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.ta"
!sed -n '100p' "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/all_data.en"

வலுவான அரசிற்கான அழைப்பு .
the call for the strong state .


In [ ]:
!ls -lh "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

total 514M
-rw------- 1 root root  36M Nov  8 04:36 all_data.en
-rw------- 1 root root 111M Nov  8 04:36 all_data.ta
-rw------- 1 root root 5.9M Nov  7 07:23 data.en1
-rw------- 1 root root 5.9M Nov  7 07:25 data.en2
-rw------- 1 root root 5.9M Nov  7 07:25 data.en3
-rw------- 1 root root 6.1M Nov  7 07:25 data.en4
-rw------- 1 root root 6.7M Nov  7 07:26 data.en5
-rw------- 1 root root 5.3M Nov  7 07:26 data.en6
-rw------- 1 root root  18M Nov  7 07:26 data.ta1
-rw------- 1 root root  19M Nov  7 07:26 data.ta2
-rw------- 1 root root  19M Nov  7 07:26 data.ta3
-rw------- 1 root root  19M Nov  7 07:26 data.ta4
-rw------- 1 root root  22M Nov  7 07:27 data.ta5
-rw------- 1 root root  17M Nov  7 07:27 data.ta6
-rw------- 1 root root 147M Nov  7 07:34 tamil_english_sentences.csv
-rw------- 1 root root  11M Nov  8 05:17 train.small.en
-rw------- 1 root root  33M Nov  8 05:29 train.small.fixed.ta
-rw------- 1 root root  33M Nov  8 05:17 train.small.ta


In [ ]:
# -----------------------------------------------
# Step 1: Data Preparation (Tamil untouched, English tokenized)
# -----------------------------------------------

import os

# Path to your dataset
path = "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

# Filenames
tamil_file = f"{path}/all_data.ta"
english_file = f"{path}/all_data.en"

#  1. Verify both files exist
assert os.path.exists(tamil_file), "Tamil file not found!"
assert os.path.exists(english_file), "English file not found!"

#  2. Read both with UTF-8
with open(tamil_file, encoding="utf-8") as f:
    tamil_lines = [line.strip() for line in f.readlines() if line.strip()]

with open(english_file, encoding="utf-8") as f:
    english_lines = [line.strip() for line in f.readlines() if line.strip()]

#  3. Ensure equal length
min_len = min(len(tamil_lines), len(english_lines))
tamil_lines = tamil_lines[:min_len]
english_lines = english_lines[:min_len]

print(f"Total aligned lines: {min_len:,}")

# 4. Take a manageable subset (100k pairs)
subset = 100000 if min_len > 100000 else min_len
tamil_subset = tamil_lines[:subset]
english_subset = english_lines[:subset]

print(f"Subset created with {subset:,} lines.")

# -----------------------------------------------
# Step 2: English Tokenization (Tamil untouched)
# -----------------------------------------------

# Install Moses tokenizer if not already done
!git clone https://github.com/moses-smt/mosesdecoder.git -q

tokenizer_script = "/content/mosesdecoder/scripts/tokenizer/tokenizer.perl"

# Tokenize English (UTF-8 safe)
os.system(f"perl {tokenizer_script} -l en < {english_file} > {path}/train.tok.en")

# Copy Tamil directly (no tokenization)
with open(f"{path}/train.tok.ta", "w", encoding="utf-8") as f:
    for line in tamil_subset:
        f.write(line + "\n")

# -----------------------------------------------
# Step 3: Visual Verification
# -----------------------------------------------
print("\n Sample Tamil (should look perfectly intact):")
!head -n 3 "$path/train.tok.ta"

print("\n Sample English (should be tokenized, e.g. commas separated):")
!head -n 3 "$path/train.tok.en"

# -----------------------------------------------
# Step 4: Final Confirmation
# -----------------------------------------------
if os.path.exists(f"{path}/train.tok.en") and os.path.exists(f"{path}/train.tok.ta"):
    print("\n Step 1 successful: Tamil preserved, English tokenized, UTF-8 maintained.")
else:
    print("\n Step 1 failed — check paths or permissions.")


In [ ]:
# -----------------------------------------------
# Step 2: Clean and Filter Corpus (Moses clean-corpus.perl)
# -----------------------------------------------
import os

path = "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

# Input tokenized files from Step 1
ta_tok = f"{path}/train.tok.ta"
en_tok = f"{path}/train.tok.en"

# Output cleaned file prefix
clean_prefix = f"{path}/train.clean"

#  Run the Moses cleaning script
os.system(f"perl /content/mosesdecoder/scripts/training/clean-corpus-n.perl "
          f"{path}/train.tok ta en {clean_prefix} 2 80")

# -----------------------------------------------
# Step 3: Verify cleaning results
# -----------------------------------------------
ta_clean = f"{path}/train.clean.ta"
en_clean = f"{path}/train.clean.en"

# Count lines
ta_lines = int(os.popen(f"wc -l < '{ta_clean}'").read().strip())
en_lines = int(os.popen(f"wc -l < '{en_clean}'").read().strip())

print(f" Cleaning complete. Tamil lines: {ta_lines:,}, English lines: {en_lines:,}")

# -----------------------------------------------
# Step 4: Preview first few cleaned sentences
# -----------------------------------------------
print("\n Sample Tamil (UTF-8 intact):")
!head -n 3 "$path/train.clean.ta"

print("\n Sample English (tokenized and cleaned):")
!head -n 3 "$path/train.clean.en"

# -----------------------------------------------
# Step 5: Final confirmation
# -----------------------------------------------
if os.path.exists(ta_clean) and os.path.exists(en_clean):
    print("\n Step 2 successful: Clean, balanced parallel corpus ready for alignment.")
else:
    print("\n Step 2 failed — check paths or script output.")


🧹 Cleaning complete. Tamil lines: 99,999, English lines: 99,999

📝 Sample Tamil (UTF-8 intact):
ராஜாவாகிய ஆகாஸ் அரசாளும்போது தம்முடைய பாதகத்தினால் எறிந்துபோட்ட சகல பணிமுட்டுகளையும் முஸ்திப்பாக்கிப் பரிசுத்தம்பண்ணினோம்; இதோ , அவைகள் கர்த்தரின் ஆலயத்திற்கு முன்பாக இருக்கிறது என்றார்கள் .
சர்வதேச நாணய நிதியம் இலங்கைக்கு கடன் வழங்கினால் இதே போன்ற நிபந்தனைகள் திணிக்கப்படும் .
தற்போது அதற்கு எதிராக வாதாடுகிறார் சர்வதேச சட்டத்தை செயல்படுத்துவதற்குப் பதிலாக புதிய சட்டம் உருவாக்கப்பட்டு நிறுவப்பட வேண்டும் என்று எழுதுகிறார் .

📝 Sample English (tokenized and cleaned):
moreover all the vessels , which king ahaz in his reign did cast away in his transgression , have we prepared and sanctified , and , behold , they are before the altar of the lord .
similar conditions will be imposed if the sri lankan government is given an imf loan .
now kornelius argues the opposite instead of enforcing the adherence to international law , new laws would now have to be devised and established .

✅ Step 2 succes

In [ ]:
# -----------------------------------------------
# Fix GIZA++ binary paths for Moses
# -----------------------------------------------

# Create directory expected by Moses
!mkdir -p /content/external_bin

# Link the GIZA++ binaries
!ln -sf /content/giza-pp/GIZA++-v2/GIZA++ /content/external_bin/GIZA++
!ln -sf /content/giza-pp/mkcls-v2/mkcls /content/external_bin/mkcls
!ln -sf /content/giza-pp/GIZA++-v2/snt2cooc.out /content/external_bin/snt2cooc.out

# Verify all exist
!echo " Linked binaries:"
!ls -lh /content/external_bin


In [ ]:
# Build a UTF-8 safe parallel file with " ||| " separator, using the cleaned files
import os

path = "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"
ta = os.path.join(path, "train.clean.ta")
en = os.path.join(path, "train.clean.en")
parallel = os.path.join(path, "parallel.txt")

with open(ta, encoding="utf-8") as f_ta, open(en, encoding="utf-8") as f_en, open(parallel, "w", encoding="utf-8") as out:
    for ta_line, en_line in zip(f_ta, f_en):
        ta_line = ta_line.strip()
        en_line = en_line.strip()
        if ta_line and en_line:
            out.write(f"{ta_line} ||| {en_line}\n")

# Quick sanity check
print(" parallel.txt created.")
!wc -l "$path/parallel.txt"
!head -n 2 "$path/parallel.txt"


✅ parallel.txt created.
99999 /content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/parallel.txt
ராஜாவாகிய ஆகாஸ் அரசாளும்போது தம்முடைய பாதகத்தினால் எறிந்துபோட்ட சகல பணிமுட்டுகளையும் முஸ்திப்பாக்கிப் பரிசுத்தம்பண்ணினோம்; இதோ , அவைகள் கர்த்தரின் ஆலயத்திற்கு முன்பாக இருக்கிறது என்றார்கள் . ||| moreover all the vessels , which king ahaz in his reign did cast away in his transgression , have we prepared and sanctified , and , behold , they are before the altar of the lord .
சர்வதேச நாணய நிதியம் இலங்கைக்கு கடன் வழங்கினால் இதே போன்ற நிபந்தனைகள் திணிக்கப்படும் . ||| similar conditions will be imposed if the sri lankan government is given an imf loan .


In [ ]:
# Run FastAlign forward (ta→en) and reverse (en→ta), then symmetrize
path="/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

! /content/external_bin/fast_align -i "$path/parallel.txt" -d -o -v  > "$path/forward.align"
! /content/external_bin/fast_align -i "$path/parallel.txt" -r -d -o -v > "$path/reverse.align"
! /content/external_bin/atools -i "$path/forward.align" -j "$path/reverse.align" -c grow-diag-final-and > "$path/symmetrized.align"

! echo " FastAlign forward & reverse complete, and alignments symmetrized."
! wc -l "$path/forward.align" "$path/reverse.align" "$path/symmetrized.align"
! head -n 3 "$path/symmetrized.align"

ARG=i
ARG=d
ARG=o
ARG=v
INITIAL PASS 
.................................................. [50000]
.................................................
expected target length = source length * 1.50628
ITERATION 1
.................................................. [50000]
.................................................
  log_e likelihood: -4.778e+07
  log_2 likelihood: -6.8932e+07
     cross entropy: 29.8974
        perplexity: 1e+09
      posterior p0: 0.08
 posterior al-feat: -0.168795
       size counts: 1764
ITERATION 2
.................................................. [50000]
.................................................
  log_e likelihood: -1.43705e+07
  log_2 likelihood: -2.07323e+07
     cross entropy: 8.99204
        perplexity: 509.184
      posterior p0: 0.0930281
 posterior al-feat: -0.144809
       size counts: 1764
  1  model al-feat: -0.112909 (tension=4)
  2  model al-feat: -0.122661 (tension=3.36199)
  3  model al-feat: -0.129905 (tension=2.91902)
  4  model al-feat: 

In [ ]:
!path="/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH" && \
idx=25 && \
echo "🔹 Tamil:" && \
sed -n "${idx}p" "$path/train.clean.ta" && \
echo "" && \
echo "🔹 English:" && \
sed -n "${idx}p" "$path/train.clean.en" && \
echo "" && \
echo "🔹 Alignment (ta→en):" && \
sed -n "${idx}p" "$path/symmetrized.align" && \
echo "" && \
echo " To check another sentence, just change the 'idx' value above and rerun."


🔹 Tamil:
சுருளைச் சம்பிரதியாகிய எலிசாமாவின் அறையிலே வைத்து , ராஜாவினிடத்துக்கு அரமனையிலே போய் , ராஜாவின் செவிகளுக்கு இந்த வார்த்தைகளையெல்லாம் அறிவித்தார்கள் .

🔹 English:
and they went in to the king into the court , but they laid up the roll in the chamber of elishama the scribe , and told all the words in the ears of the king .

🔹 Alignment (ta→en):
0-1 1-23 2-21 3-19 4-11 5-10 6-15 6-16 6-18 7-3 7-17 8-2 8-4 8-25 9-24 10-35 12-28 13-27 13-29 14-26 15-36

✅ To check another sentence, just change the 'idx' value above and rerun.


In [ ]:
# -----------------------------------------------
#  Step 4: Build Tamil→English Word Dictionary
# -----------------------------------------------
from collections import defaultdict, Counter
import pandas as pd

path = "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"

# Load corpus and alignments
with open(f"{path}/train.clean.ta", encoding="utf-8") as f_ta, \
     open(f"{path}/train.clean.en", encoding="utf-8") as f_en, \
     open(f"{path}/symmetrized.align", encoding="utf-8") as f_align:

    tamil_lines = f_ta.readlines()
    english_lines = f_en.readlines()
    align_lines = f_align.readlines()

# Dictionary: tamil_word → Counter({english_word: count})
bilingual_dict = defaultdict(Counter)

# Iterate through each aligned line
for ta_sent, en_sent, align in zip(tamil_lines, english_lines, align_lines):
    ta_words = ta_sent.strip().split()
    en_words = en_sent.strip().split()

    for pair in align.strip().split():
        if "-" not in pair:
            continue
        ta_idx, en_idx = map(int, pair.split("-"))
        if ta_idx < len(ta_words) and en_idx < len(en_words):
            bilingual_dict[ta_words[ta_idx]][en_words[en_idx]] += 1

# Convert to DataFrame (top 3 English translations per Tamil word)
data = []
for ta_word, en_counts in bilingual_dict.items():
    for en_word, count in en_counts.most_common(3):
        data.append((ta_word, en_word, count))

df = pd.DataFrame(data, columns=["Tamil_Word", "English_Word", "Frequency"])

# Save to Drive
out_path = f"{path}/tamil_english_word_dictionary.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f" Bilingual dictionary created and saved as:\n{out_path}")
print(f"Total Tamil words mapped: {len(df.Tamil_Word.unique()):,}")
print("\n🪄 Sample entries:")
print(df.sample(10))


✅ Bilingual dictionary created and saved as:
/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH/tamil_english_word_dictionary.csv
Total Tamil words mapped: 208,110

🪄 Sample entries:
              Tamil_Word English_Word  Frequency
229378       உப்பளத்தில்          the          1
82375             199397        since          1
255885    பொதுக்கடன்கள்     increase          1
35644           விவகாரம்       affair         16
106903         ஹுசேனுடைய    aftermath          1
92070              துக்க         with          1
301359           ஏதையும்      execute          1
131027              பவர்         know          1
282250  நெருக்கடியாகும்;       crisis          1
22850     ஆபிரகாமுக்குப்      abraham          2


In [ ]:
import pandas as pd

# Load the bilingual dictionary
path = "/content/drive/MyDrive/DATASETS_MLM_PROJECT/FINAL/FINAL_DATASETS/TAMIL-ENGLISH"
df = pd.read_csv(f"{path}/tamil_english_word_dictionary.csv")

# Drop any empty/NaN entries
df = df.dropna(subset=["Tamil_Word", "English_Word"])

# Create dictionary: Tamil → most frequent English
tamil_to_english = (
    df.groupby("Tamil_Word")["English_Word"]
    .agg(lambda x: x.value_counts().idxmax() if not x.empty else "[UNK]")
    .to_dict()
)

# 🔹 Test Tamil sentence
tamil_sentence = "அவர் அரசாங்கத்தில் முக்கியமான பதவியில் உள்ளார்"

# Tokenize Tamil sentence
tamil_words = tamil_sentence.strip().split()

# Map to English equivalents
mapped_english = [
    tamil_to_english.get(word, "[UNK]") for word in tamil_words
]

# Display results
print("🔹 Tamil Sentence:")
print(" ".join(tamil_words))
print("\n🔹 Mapped English Words:")
print(" ".join(mapped_english))


🔹 Tamil Sentence:
அவர் அரசாங்கத்தில் முக்கியமான பதவியில் உள்ளார்

🔹 Mapped English Words:
he government important in .
